<a href="https://colab.research.google.com/github/bangaru01/C_programing/blob/main/Input_orientation_log_to_XYZ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 새 섹션

In [10]:
# ============================================================
# GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive', force_remount=True)

import os


# ============================================================
# PATHS
# ============================================================

log_folder = "/content/drive/MyDrive/log-files"
xyz_folder = "/content/drive/MyDrive/xyz_files"


# Check LOG folder
if not os.path.isdir(log_folder):

    raise FileNotFoundError(
        f"LOG folder not found:\n{log_folder}"
    )


# Create XYZ folder if needed
os.makedirs(
    xyz_folder,
    exist_ok=True
)


print("Google Drive mounted")
print("LOG folder:", log_folder)
print("XYZ folder:", xyz_folder)


# ============================================================
# ATOMIC NUMBER → ELEMENT SYMBOL
# ============================================================

elements = [
    "",
    "H", "He", "Li", "Be", "B", "C", "N", "O", "F",
    "Ne", "Na", "Mg", "Al", "Si", "P", "S", "Cl", "Ar",
    "K", "Ca", "Sc", "Ti", "V", "Cr", "Mn", "Fe",
    "Co", "Ni", "Cu", "Zn", "Ga", "Ge", "As", "Se",
    "Br", "Kr", "Rb", "Sr", "Y", "Zr", "Nb", "Mo",
    "Tc", "Ru", "Rh", "Pd", "Ag", "Cd", "In", "Sn",
    "Sb", "Te", "I", "Xe"
]


atomic_symbol = {
    i: element
    for i, element in enumerate(elements)
}


# ============================================================
# EXTRACT INPUT ORIENTATION BLOCKS
# ============================================================

def extract_input_orientation_blocks(lines):

    blocks = []

    i = 0

    while i < len(lines):

        # ----------------------------------------------------
        # Find "Input orientation:"
        # ----------------------------------------------------

        if "Input orientation:" not in lines[i]:

            i += 1
            continue


        # ----------------------------------------------------
        # Find first dashed line
        # ----------------------------------------------------

        start = i + 1

        while start < len(lines):

            if lines[start].strip().startswith("-----"):

                break

            start += 1


        # ----------------------------------------------------
        # Move to first atom line
        # ----------------------------------------------------

        start += 1


        atoms = []


        # ----------------------------------------------------
        # Read atomic coordinates
        # ----------------------------------------------------

        while start < len(lines):

            line = lines[start].strip()


            # End of coordinate table
            if line.startswith("-----"):

                break


            parts = line.split()


            # Gaussian format:
            #
            # Center
            # Atomic Number
            # Atomic Type
            # X
            # Y
            # Z
            #
            # Example:
            #
            # 1  6  0  -0.123456  1.234567  0.456789

            if len(parts) >= 6:

                try:

                    atomic_number = int(parts[1])

                    x = float(parts[3])

                    y = float(parts[4])

                    z = float(parts[5])


                    atoms.append(
                        (
                            atomic_number,
                            x,
                            y,
                            z
                        )
                    )


                except ValueError:

                    pass


            start += 1


        # ----------------------------------------------------
        # Save this coordinate block
        # ----------------------------------------------------

        if atoms:

            blocks.append(atoms)


        i = start + 1


    return blocks


# ============================================================
# CONVERT ONE LOG FILE → XYZ
# ============================================================

def convert_log_to_xyz(log_file):

    log_path = os.path.join(
        log_folder,
        log_file
    )


    xyz_name = (
        os.path.splitext(log_file)[0]
        + ".xyz"
    )


    xyz_path = os.path.join(
        xyz_folder,
        xyz_name
    )


    # --------------------------------------------------------
    # Read LOG file
    # --------------------------------------------------------

    with open(
        log_path,
        "r",
        errors="ignore"
    ) as f:

        lines = f.readlines()


    # --------------------------------------------------------
    # Extract Input orientation blocks
    # --------------------------------------------------------

    blocks = extract_input_orientation_blocks(
        lines
    )


    # --------------------------------------------------------
    # Check coordinates
    # --------------------------------------------------------

    if not blocks:

        print(
            f"❌ {log_file}: "
            "No Input orientation coordinates found"
        )

        return False


    # --------------------------------------------------------
    # IMPORTANT:
    #
    # Use FINAL Input orientation block
    # --------------------------------------------------------

    atoms = blocks[-1]


    # --------------------------------------------------------
    # Write XYZ
    # --------------------------------------------------------

    with open(
        xyz_path,
        "w"
    ) as fout:


        # Number of atoms
        fout.write(
            f"{len(atoms)}\n"
        )


        # Comment / molecule name
        name = os.path.splitext(
            log_file
        )[0]


        fout.write(
            f"{name}\n"
        )


        # Coordinates
        for (
            atomic_number,
            x,
            y,
            z
        ) in atoms:


            symbol = atomic_symbol.get(
                atomic_number,
                "X"
            )


            fout.write(
                f"{symbol:2s} "
                f"{x:14.8f} "
                f"{y:14.8f} "
                f"{z:14.8f}\n"
            )


    # --------------------------------------------------------
    # Report
    # --------------------------------------------------------

    print(
        f"✅ Converted: {log_file}"
    )


    print(
        f"   Coordinate blocks found: {len(blocks)}"
    )


    print(
        f"   Final geometry atoms: {len(atoms)}"
    )


    print(
        f"   XYZ saved as: {xyz_name}"
    )


    return True


# ============================================================
# FIND ALL LOG FILES
# ============================================================

log_files = sorted(
    [
        f
        for f in os.listdir(log_folder)
        if f.lower().endswith(".log")
    ]
)


print("\n" + "=" * 70)

print(
    "🔄 CONVERTING LOG → XYZ"
)

print("=" * 70)


print(
    f"\nFound {len(log_files)} LOG file(s)\n"
)


# ============================================================
# CONVERT ALL LOG FILES
# ============================================================

converted = 0

failed = 0


for log_file in log_files:


    success = convert_log_to_xyz(
        log_file
    )


    if success:

        converted += 1

    else:

        failed += 1


# ============================================================
# CREATE COMBINED SI.txt
# ============================================================

print("\n" + "=" * 70)

print(
    "📄 CREATING SI.txt"
)

print("=" * 70)


si_file = os.path.join(
    xyz_folder,
    "SI.txt"
)


# Get XYZ files
xyz_files = sorted(
    [
        f
        for f in os.listdir(xyz_folder)
        if f.lower().endswith(".xyz")
    ]
)


# ------------------------------------------------------------
# Write SI.txt
# ------------------------------------------------------------

with open(
    si_file,
    "w"
) as fout:


    for xyz_file in xyz_files:


        xyz_path = os.path.join(
            xyz_folder,
            xyz_file
        )


        with open(
            xyz_path,
            "r",
            errors="ignore"
        ) as fin:

            content = fin.read()


        name = os.path.splitext(
            xyz_file
        )[0]


        fout.write(
            "=" * 70
            + "\n"
        )


        fout.write(
            name
            + "\n"
        )


        fout.write(
            "=" * 70
            + "\n"
        )


        fout.write(
            content
        )


        fout.write(
            "\n\n"
        )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)

print(
    "🎉 DONE"
)

print("=" * 70)


print(
    f"Converted : {converted}"
)


print(
    f"Failed    : {failed}"
)


print(
    f"XYZ files : {xyz_folder}"
)


print(
    f"SI file   : {si_file}"
)


print(
    f"\nXYZ files included in SI.txt: {len(xyz_files)}"
)


# ============================================================
# LIST XYZ FILES
# ============================================================

print("\n📂 XYZ files:")


for xyz_file in xyz_files:

    print(
        "   ",
        xyz_file
    )

Mounted at /content/drive
Google Drive mounted
LOG folder: /content/drive/MyDrive/log-files
XYZ folder: /content/drive/MyDrive/xyz_files

🔄 CONVERTING LOG → XYZ

Found 1 LOG file(s)

❌ Sub3g_TS2-1_RR-cis-olefin-Hs_type-1-Aug18-1_TS.log: No Input orientation coordinates found

📄 CREATING SI.txt

🎉 DONE
Converted : 0
Failed    : 1
XYZ files : /content/drive/MyDrive/xyz_files
SI file   : /content/drive/MyDrive/xyz_files/SI.txt

XYZ files included in SI.txt: 0

📂 XYZ files:


In [12]:
# ============================================================
# GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import re


# ============================================================
# PATHS
# ============================================================

log_folder = "/content/drive/MyDrive/log-files"
xyz_folder = "/content/drive/MyDrive/xyz_files"

if not os.path.isdir(log_folder):
    raise FileNotFoundError(
        f"LOG folder not found:\n{log_folder}"
    )

os.makedirs(xyz_folder, exist_ok=True)


print("✅ Google Drive mounted")
print("📂 LOG folder:", log_folder)
print("📁 XYZ folder:", xyz_folder)


# ============================================================
# ATOMIC NUMBER → SYMBOL
# ============================================================

elements = [
    "",
    "H","He","Li","Be","B","C","N","O","F",
    "Ne","Na","Mg","Al","Si","P","S","Cl","Ar",
    "K","Ca","Sc","Ti","V","Cr","Mn","Fe",
    "Co","Ni","Cu","Zn","Ga","Ge","As","Se",
    "Br","Kr","Rb","Sr","Y","Zr","Nb","Mo",
    "Tc","Ru","Rh","Pd","Ag","Cd","In","Sn",
    "Sb","Te","I","Xe"
]


# ============================================================
# FUNCTION:
# EXTRACT ALL INPUT ORIENTATION BLOCKS
# ============================================================

def extract_input_orientation(lines):

    blocks = []

    i = 0

    while i < len(lines):

        # ----------------------------------------------------
        # Find Input orientation
        # ----------------------------------------------------

        if "Input orientation:" not in lines[i]:

            i += 1
            continue


        atoms = []

        j = i + 1

        # ----------------------------------------------------
        # Search forward for coordinate rows
        # ----------------------------------------------------

        while j < len(lines):

            line = lines[j].strip()

            # Stop if another major Gaussian section starts
            if (
                j > i + 5
                and (
                    "SCF Done:" in line
                    or "Optimization completed" in line
                    or "Stationary point found" in line
                )
            ):
                break


            parts = line.split()


            # ------------------------------------------------
            # Gaussian coordinate format:
            #
            # Center AtomicNumber AtomicType X Y Z
            #
            # Example:
            #
            # 1 6 0 -0.123456 1.234567 2.345678
            # ------------------------------------------------

            if len(parts) >= 6:

                try:

                    center = int(parts[0])
                    atomic_number = int(parts[1])
                    atomic_type = int(parts[2])

                    x = float(parts[3])
                    y = float(parts[4])
                    z = float(parts[5])


                    # Basic validation
                    if (
                        center > 0
                        and atomic_number > 0
                        and atomic_number < len(elements)
                    ):

                        atoms.append(
                            (
                                atomic_number,
                                x,
                                y,
                                z
                            )
                        )


                except (ValueError, IndexError):

                    pass


            # ------------------------------------------------
            # End after coordinates followed by dashed line
            # ------------------------------------------------

            if atoms and line.startswith("-----"):

                break


            j += 1


        # ----------------------------------------------------
        # Save block
        # ----------------------------------------------------

        if atoms:

            blocks.append(atoms)


        i = j + 1


    return blocks


# ============================================================
# FUNCTION:
# CONVERT ONE LOG FILE
# ============================================================

def convert_log(log_file):

    log_path = os.path.join(
        log_folder,
        log_file
    )


    print("\n" + "-" * 70)
    print("Processing:", log_file)
    print("-" * 70)


    # --------------------------------------------------------
    # Read file
    # --------------------------------------------------------

    with open(
        log_path,
        "r",
        errors="ignore"
    ) as f:

        lines = f.readlines()


    print(
        "Total lines:",
        len(lines)
    )


    # --------------------------------------------------------
    # Count Input orientation
    # --------------------------------------------------------

    orientation_count = sum(
        "Input orientation:" in line
        for line in lines
    )


    print(
        "Input orientation sections:",
        orientation_count
    )


    # --------------------------------------------------------
    # Extract coordinates
    # --------------------------------------------------------

    blocks = extract_input_orientation(
        lines
    )


    print(
        "Coordinate blocks extracted:",
        len(blocks)
    )


    # --------------------------------------------------------
    # Check
    # --------------------------------------------------------

    if not blocks:

        print(
            "❌ NO COORDINATES EXTRACTED"
        )

        return False


    # --------------------------------------------------------
    # Use FINAL geometry
    # --------------------------------------------------------

    atoms = blocks[-1]


    print(
        "Final geometry atoms:",
        len(atoms)
    )


    # ========================================================
    # WRITE XYZ
    # ========================================================

    xyz_name = (
        os.path.splitext(log_file)[0]
        + ".xyz"
    )


    xyz_path = os.path.join(
        xyz_folder,
        xyz_name
    )


    with open(
        xyz_path,
        "w"
    ) as fout:

        # Number of atoms
        fout.write(
            str(len(atoms))
            + "\n"
        )


        # Comment
        fout.write(
            os.path.splitext(log_file)[0]
            + "\n"
        )


        # Coordinates
        for (
            atomic_number,
            x,
            y,
            z
        ) in atoms:


            symbol = elements[
                atomic_number
            ]


            fout.write(
                f"{symbol:2s} "
                f"{x:14.8f} "
                f"{y:14.8f} "
                f"{z:14.8f}\n"
            )


    print(
        "✅ XYZ created:"
    )

    print(
        xyz_path
    )


    return True


# ============================================================
# FIND ALL LOG FILES
# ============================================================

log_files = sorted(
    [
        f
        for f in os.listdir(log_folder)
        if f.lower().endswith(".log")
    ]
)


print("\n" + "=" * 70)
print("🔄 LOG → XYZ CONVERSION")
print("=" * 70)

print(
    f"\nFound {len(log_files)} LOG file(s)"
)


# ============================================================
# CONVERT
# ============================================================

converted = 0
failed = 0


for log_file in log_files:

    try:

        success = convert_log(
            log_file
        )


        if success:

            converted += 1

        else:

            failed += 1


    except Exception as error:

        failed += 1

        print(
            "❌ ERROR:",
            error
        )


# ============================================================
# CREATE SI.txt
# ============================================================

print("\n" + "=" * 70)
print("📄 CREATING SI.txt")
print("=" * 70)


xyz_files = sorted(
    [
        f
        for f in os.listdir(xyz_folder)
        if f.lower().endswith(".xyz")
    ]
)


si_file = os.path.join(
    xyz_folder,
    "SI.txt"
)


with open(
    si_file,
    "w"
) as fout:

    for xyz_file in xyz_files:

        xyz_path = os.path.join(
            xyz_folder,
            xyz_file
        )


        with open(
            xyz_path,
            "r",
            errors="ignore"
        ) as fin:

            content = fin.read()


        name = os.path.splitext(
            xyz_file
        )[0]


        fout.write(
            "=" * 70 + "\n"
        )

        fout.write(
            name + "\n"
        )

        fout.write(
            "=" * 70 + "\n"
        )

        fout.write(
            content
        )

        fout.write(
            "\n\n"
        )


# ============================================================
# FINAL REPORT
# ============================================================

print("\n" + "=" * 70)
print("🎉 FINAL RESULT")
print("=" * 70)

print(
    f"LOG files found : {len(log_files)}"
)

print(
    f"Converted       : {converted}"
)

print(
    f"Failed          : {failed}"
)

print(
    f"XYZ folder      : {xyz_folder}"
)

print(
    f"SI.txt          : {si_file}"
)


print("\n📂 XYZ files:")

for xyz_file in xyz_files:

    print(
        "   ",
        xyz_file
    )

Mounted at /content/drive
✅ Google Drive mounted
📂 LOG folder: /content/drive/MyDrive/log-files
📁 XYZ folder: /content/drive/MyDrive/xyz_files

🔄 LOG → XYZ CONVERSION

Found 34 LOG file(s)

----------------------------------------------------------------------
Processing: RR_CH2-Ph.log
----------------------------------------------------------------------
Total lines: 48034
Input orientation sections: 73
Coordinate blocks extracted: 73
Final geometry atoms: 57
✅ XYZ created:
/content/drive/MyDrive/xyz_files/RR_CH2-Ph.xyz

----------------------------------------------------------------------
Processing: RR_CH2-dioxane.log
----------------------------------------------------------------------
Total lines: 62331
Input orientation sections: 100
Coordinate blocks extracted: 100
Final geometry atoms: 56
✅ XYZ created:
/content/drive/MyDrive/xyz_files/RR_CH2-dioxane.xyz

----------------------------------------------------------------------
Processing: RR_CH2CN.log
--------------------------